# Eigenvalues and Eigenvectors

An eigenvector of $A$ is a direction that $A$ only stretches: $Av = \lambda v$. This notebook computes eigenvalues, uses them to predict the long-run behaviour of a linear recursion, and finds the largest one with the power method.

In [ ]:
Pollis.packages("Plots")

In [ ]:
using LinearAlgebra, Plots

## A labour-market transition matrix

Each month a worker is employed, unemployed or out of the labour force, and moves between these states with the probabilities below (made up for illustration). Column $j$ holds the probabilities of moving *from* state $j$, so each column sums to 1, and the distribution evolves as $p_{t+1} = P p_t$.

In [ ]:
P = [0.95 0.30 0.05;
     0.03 0.60 0.05;
     0.02 0.10 0.90]   # from: employed, unemployed, out of the labour force
(column_sums = vec(sum(P, dims=1)),)

In [ ]:
E = eigen(P)
E.values

A transition matrix always has the eigenvalue 1, and every other eigenvalue has modulus below 1. Along the eigenvector of $\lambda = 1$ nothing changes: it is the long-run distribution. The others decay, the slowest at rate $\lvert\lambda_2\rvert$ per month.

In [ ]:
j = argmax(real.(E.values))
v = real.(E.vectors[:, j])
stationary = v / sum(v)
(long_run_shares = round.(stationary, digits=4), slowest_decay = sort(abs.(E.values))[end - 1])

Iterate from everyone employed and watch the distribution settle.

In [ ]:
p = [1.0, 0, 0]
path = [p]
for t in 1:60
    push!(path, P * path[end])
end
Pm = reduce(hcat, path)
plot(0:60, transpose(Pm), label=["employed" "unemployed" "out of labour force"], xlabel="month", ylabel="share", lw=2)
hline!(stationary, ls=:dot, c=:gray, label="")

## The power method

Multiplying any starting vector by $A$ over and over turns it towards the eigenvector of the eigenvalue largest in modulus; the stretch per step tends to that eigenvalue.

In [ ]:
A = [4.0 1 2; 1 5 1; 2 1 6]
x = ones(3)
for k in 1:15
    y = A * x
    global growth = norm(y) / norm(x)
    global x = y / norm(y)
end
(power_method = growth, largest_eigenvalue = maximum(eigvals(A)))

## Symmetric matrices and curvature

For a symmetric matrix the eigenvalues are real and the eigenvectors orthogonal. Their signs give the shape of $x^\top A x$: all positive is a bowl (positive definite), mixed signs a saddle.

In [ ]:
H1 = [2.0 0.5; 0.5 1.0]
H2 = [1.0 2.0; 2.0 1.0]
(bowl = eigvals(Symmetric(H1)), saddle = eigvals(Symmetric(H2)))

## References

[Francis (1961)](https://doi.org/10.1093/comjnl/4.3.265), The QR transformation: a unitary analogue to the LR transformation, part 1, *The Computer Journal* 4(3), 265–271.